### Load Data & Feature Stacking (TF-IDF)

In [7]:
import pandas as pd
import time
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split

print("Loading datasets...")
df_clean = pd.read_csv('../../data_resource/preprocessed_recipes.csv')
df_raw = pd.read_csv('../../data_resource/recipes.csv', usecols=['RecipeId', 'RecipeCategory', 'RecipeIngredientParts'])

df = pd.merge(df_clean, df_raw, on='RecipeId')
df = df.dropna(subset=['RecipeCategory'])

top_categories = df['RecipeCategory'].value_counts().nlargest(5).index
df_top = df[df['RecipeCategory'].isin(top_categories)].copy()
print(f"Filtered top 5 categories: {len(df_top)} rows.")

print("Extracting features (Title, Body, and Char-level TF-IDF)...")
start_time = time.time()

tfidf_title = TfidfVectorizer(max_features=10000)
X_title = tfidf_title.fit_transform(df_top['Name'].fillna(''))

tfidf_body = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)
X_body = tfidf_body.fit_transform(df_top['RecipeIngredientParts'].fillna(''))

tfidf_char = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=10000)
X_char = tfidf_char.fit_transform(df_top['Name'].fillna(''))

X = hstack([X_title, X_body, X_char])
y = df_top['RecipeCategory']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Feature stacking completed in {time.time() - start_time:.2f} seconds.")

Loading datasets...
Filtered top 5 categories: 174335 rows.
Extracting features (Title, Body, and Char-level TF-IDF)...
Feature stacking completed in 8.40 seconds.


### Hyperparameter Tuning with Optuna (50 Trials)

In [8]:
import optuna
import time
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

print("Starting Optuna optimization (50 trials) with SVM (Hinge Loss) and balanced class weights...")

def objective(trial):
    alpha = trial.suggest_float("alpha", 1e-6, 1e-3, log=True)
    penalty = trial.suggest_categorical("penalty", ["l2", "elasticnet"])

    clf = SGDClassifier(loss='hinge', alpha=alpha, penalty=penalty, class_weight='balanced', max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    score = f1_score(y_test, y_pred, average='macro')
    return score

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="maximize")
start_time = time.time()
study.optimize(objective, n_trials=50)

print(f"Optimization finished in {time.time() - start_time:.2f} seconds.")
print(f"Best Trial F1-Score: {study.best_value:.4f}")
print(f"Best Parameters: {study.best_params}")

Starting Optuna optimization (50 trials) with SVM (Hinge Loss) and balanced class weights...
Optimization finished in 421.53 seconds.
Best Trial F1-Score: 0.8129
Best Parameters: {'alpha': 1.946128980175487e-05, 'penalty': 'elasticnet'}


### Train Final Model & Save

In [9]:
import pickle
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score

print("Training final model with best parameters (SVM Hinge Loss)...")
best_params = study.best_params

final_clf = SGDClassifier(
    loss='hinge',
    alpha=best_params['alpha'],
    penalty=best_params['penalty'],
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
final_clf.fit(X_train, y_train)

y_pred_final = final_clf.predict(X_test)
final_f1 = f1_score(y_test, y_pred_final, average='macro')
print(f" Final Model F1-Score: {final_f1:.4f}")

print("Exporting ML models and vectorizers...")
model_artifacts = {
    'classifier': final_clf,
    'tfidf_title': tfidf_title,
    'tfidf_body': tfidf_body,
    'tfidf_char': tfidf_char,
    'categories': top_categories.tolist()
}

with open('../models/recipe_classifier.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print(" All artifacts saved to '../models/recipe_classifier.pkl'. Ready for Flask!")

Training final model with best parameters (SVM Hinge Loss)...
 Final Model F1-Score: 0.8129
Exporting ML models and vectorizers...
 All artifacts saved to '../models/recipe_classifier.pkl'. Ready for Flask!
